# CONDOR FlexDC Model V3 Training — Sweep V3

This notebook trains **Model V3** on the Sweep V3 dataset using the preassigned 70/20/10 split, seed-group-aware sampling, repeat-group normalization, and one-time untouched test evaluation.

The Set-Transformer architecture is inherited from the successful behavior-model lineage, while the Sweep V3 data and evaluation contract define this trained artifact as **Model V3**.


## 0. Environment and repository controls — RUN FIRST

In [ ]:
from pathlib import Path
import os

RUN_ENV = "colab"  # "colab" or "local_pc"
WORKSPACE = Path("/content/workspace") if RUN_ENV == "colab" else Path.cwd().parent

COMDER_REPO_URL = "https://github.com/NetherMoon/CONDOR-FLEXDC.git"
COMDER_BRANCH = "main"
FORCE_RECLONE = False

USE_WANDB = True
WANDB_MODE = "online"  # "online", "offline", or "disabled"
WANDB_PROJECT = "flexdc-unified-training"
WANDB_ENTITY = "amenon06-boston-university"

# Sweep V3 is the intended production dataset for this notebook.
DATASET_PROFILE = "sweep_v3"  # "sweep_v3", "old_plus_w2dense", "sweep_v2", "custom"

# For Colab, copy the final vetted files into this expected directory or use
# the optional Google Drive cell below.
SWEEP_V3_DATA_DIR_OVERRIDE = None
# Example:
# SWEEP_V3_DATA_DIR_OVERRIDE = Path("/content/flexdc_sweep_v3_vetted")

print("RUN_ENV:", RUN_ENV)
print("WORKSPACE:", WORKSPACE)
print("DATASET_PROFILE:", DATASET_PROFILE)
print("W&B:", USE_WANDB, WANDB_MODE)


## 1. Install dependencies — COLAB ONLY

In [ ]:
if RUN_ENV == "colab":
    %pip install -q wandb pandas numpy scipy scikit-learn matplotlib tabulate openpyxl

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone or update CONDOR-FLEXDC — COLAB ONLY

In [ ]:
import subprocess

if RUN_ENV == "colab":
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    COMDER_ROOT = WORKSPACE / "comder-main"
    if FORCE_RECLONE and COMDER_ROOT.exists():
        subprocess.run(["rm", "-rf", str(COMDER_ROOT)], check=True)
    if not COMDER_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--branch", COMDER_BRANCH, COMDER_REPO_URL, str(COMDER_ROOT)],
            check=True,
        )
    else:
        print("Repository already exists; leaving local files unchanged:", COMDER_ROOT)
else:
    COMDER_ROOT = Path(r"C:/Users/Achuthan Menon/Desktop/Research Work/comder-main")

print("COMDER_ROOT:", COMDER_ROOT)

## 3. Optional Google Drive copy — RUN ONLY WHEN NEEDED

The Sweep V3 data is intentionally not assumed to be in GitHub. Copy the vetted Sweep V3 CSV, final audit JSON, and the four Model V3 source files into the paths below, or place them there manually before running the required-file check.

In [ ]:
USE_GOOGLE_DRIVE_DATA = False

# Edit only when USE_GOOGLE_DRIVE_DATA=True.
DRIVE_SWEEP_V3_TRAINING_READY = "/content/drive/MyDrive/path/to/flexdc_sweep_v3_training_ready.csv"
DRIVE_SWEEP_V3_FINAL_AUDIT = "/content/drive/MyDrive/path/to/flexdc_sweep_v3_final_audit.json"

# Optional source-file overrides. Usually the repository already contains
# these current files, so leave them blank unless you need to copy a patch.
DRIVE_MODEL_V3 = ""
DRIVE_UTILS_V3 = ""
DRIVE_TESTS_V3 = ""

if RUN_ENV == "colab" and USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive
    import shutil

    drive.mount("/content/drive")
    sweep_v3_dir = COMDER_ROOT / "am_flexdc" / "data" / "pilots" / "flexdc_sweep_v3_paper_objective"
    train_dir = COMDER_ROOT / "am_flexdc" / "train"
    sweep_v3_dir.mkdir(parents=True, exist_ok=True)
    train_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(
        DRIVE_SWEEP_V3_TRAINING_READY,
        sweep_v3_dir / "flexdc_sweep_v3_training_ready.csv",
    )
    shutil.copy2(
        DRIVE_SWEEP_V3_FINAL_AUDIT,
        sweep_v3_dir / "flexdc_sweep_v3_final_audit.json",
    )

    for source, destination in [
        (DRIVE_MODEL_V3, train_dir / "data_center_model_flexdc_behavior_v3.py"),
        (DRIVE_UTILS_V3, train_dir / "am_flexdc_behavior_training_utilities_v3.py"),
        (DRIVE_TESTS_V3, train_dir / "test_flexdc_behavior_training_v3.py"),
    ]:
        if source:
            shutil.copy2(source, destination)

    print("Copied Sweep V3 data and optional training files from Drive.")
else:
    print("Drive copy skipped.")


## 4. Resolve dataset paths and check required files

In [ ]:
import sys
import json

AM_FLEXDC_ROOT = COMDER_ROOT / "am_flexdc"
TRAIN_DIR = AM_FLEXDC_ROOT / "train"
MODELS_DIR = AM_FLEXDC_ROOT / "models" / "flexdc_behavior_v3"
RESULTS_DIR = AM_FLEXDC_ROOT / "results" / "training_runs"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

profiles = {
    "sweep_v3": {
        "pilot_dir": (
            Path(SWEEP_V3_DATA_DIR_OVERRIDE)
            if SWEEP_V3_DATA_DIR_OVERRIDE is not None
            else AM_FLEXDC_ROOT / "data" / "pilots" / "flexdc_sweep_v3_paper_objective"
        ),
        "results_name": "flexdc_sweep_v3_training_ready.csv",
        "diagnostics_name": None,
        "audit_name": "flexdc_sweep_v3_final_audit.json",
        "deduplicate": False,
        "tag": "sweep_v3_behavior_v3",
        "split_mode": "preassigned",
        "split_column": "Data_Split",
    },
    "old_plus_w2dense": {
        "pilot_dir": AM_FLEXDC_ROOT / "data" / "pilots" / "traditionaliso_newqos_pilot_plus_w2_dense_v1_flexdc_configured_objective",
        "results_name": "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_results.csv",
        "diagnostics_name": "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_diagnostics.csv",
        "audit_name": None,
        "deduplicate": False,
        "tag": "old_plus_w2dense_behavior_v3",
        "split_mode": "generated",
        "split_column": "Data_Split",
    },
    "sweep_v2": {
        "pilot_dir": AM_FLEXDC_ROOT / "data" / "pilots" / "flexdc_sweep_v2_paper_objective",
        "results_name": "flexdc_sweep_combined_grid_search_results.csv",
        "diagnostics_name": "flexdc_sweep_combined_grid_search_diagnostics.csv",
        "audit_name": None,
        "deduplicate": True,
        "tag": "sweep_v2_behavior_v3",
        "split_mode": "generated",
        "split_column": "Data_Split",
    },
}

if DATASET_PROFILE == "custom":
    PILOT_DIR = Path("/content/path/to/custom/vetted/dataset")
    RESULTS_CSV = PILOT_DIR / "training_ready.csv"
    DIAGNOSTICS_CSV = None
    FINAL_AUDIT_JSON = None
    DEDUPLICATE = False
    DATASET_TAG = "custom_behavior_v3"
    SPLIT_MODE = "auto"
    SPLIT_COLUMN = "Data_Split"
else:
    profile = profiles[DATASET_PROFILE]
    PILOT_DIR = Path(profile["pilot_dir"])
    RESULTS_CSV = PILOT_DIR / profile["results_name"]
    DIAGNOSTICS_CSV = (
        PILOT_DIR / profile["diagnostics_name"]
        if profile["diagnostics_name"]
        else None
    )
    FINAL_AUDIT_JSON = (
        PILOT_DIR / profile["audit_name"]
        if profile["audit_name"]
        else None
    )
    DEDUPLICATE = bool(profile["deduplicate"])
    DATASET_TAG = profile["tag"]
    SPLIT_MODE = profile["split_mode"]
    SPLIT_COLUMN = profile["split_column"]

MODEL_PY = TRAIN_DIR / "data_center_model_flexdc_behavior_v3.py"
UTILS_PY = TRAIN_DIR / "am_flexdc_behavior_training_utilities_v3.py"
TEST_PY = TRAIN_DIR / "test_flexdc_behavior_training_v3.py"

required = [MODEL_PY, UTILS_PY, TEST_PY, RESULTS_CSV]
if DIAGNOSTICS_CSV is not None:
    required.append(DIAGNOSTICS_CSV)
if FINAL_AUDIT_JSON is not None:
    required.append(FINAL_AUDIT_JSON)

missing = [path for path in required if not path.exists()]
if missing:
    print("Missing required files:")
    for path in missing:
        print(" -", path)
    raise FileNotFoundError("Copy the missing files before continuing.")

V3_AUDIT = None
if FINAL_AUDIT_JSON is not None:
    V3_AUDIT = json.loads(FINAL_AUDIT_JSON.read_text())
    if V3_AUDIT.get("status") != "PASS":
        raise RuntimeError(
            "Sweep V3 final audit is not PASS. Do not train on an incomplete or failed dataset."
        )
    if V3_AUDIT.get("hard_errors"):
        raise RuntimeError(f"Sweep V3 audit contains hard errors: {V3_AUDIT['hard_errors']}")
    print("Sweep V3 audit status: PASS")
    print("Audited training-ready rows:", V3_AUDIT.get("training_ready_rows"))
    if V3_AUDIT.get("warnings"):
        print("Documented audit warnings:")
        for warning in V3_AUDIT["warnings"]:
            print(" -", warning)

if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

print("All required files found.")
print("RESULTS_CSV:", RESULTS_CSV)
print("DIAGNOSTICS_CSV:", DIAGNOSTICS_CSV)
print("FINAL_AUDIT_JSON:", FINAL_AUDIT_JSON)
print("SPLIT_MODE:", SPLIT_MODE)
print("DATASET_TAG:", DATASET_TAG)


## 5. Compile and run structural tests — REQUIRED

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "py_compile", MODEL_PY.name, UTILS_PY.name, TEST_PY.name],
    cwd=TRAIN_DIR,
    check=True,
)
subprocess.run(
    [sys.executable, TEST_PY.name],
    cwd=TRAIN_DIR,
    check=True,
)
print("Compilation and structural tests passed.")

## 6. Import model and utilities

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

from data_center_model_flexdc_behavior_v3 import (
    DataCenterBehaviorModel,
    FlexDCBehaviorModelConfig,
)
from am_flexdc_behavior_training_utilities_v3 import (
    FlexDCBehaviorConstants,
    prepare_behavior_data,
    train_behavior_model,
    final_behavior_evaluation,
    final_behavior_test_evaluation,
    evaluate_behavior_loader,
    load_behavior_model_checkpoint,
    context_metrics_table,
    sample_prediction_rows,
    choose_device,
)


def show_table(df, caption=None, precision=4):
    styled = (
        df.style.hide(axis="index")
        .format(precision=precision)
        .set_properties(**{
            "text-align": "left",
            "white-space": "normal",
            "font-size": "12px",
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#0f172a"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #cbd5e1"),
                    ("padding", "6px"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                ],
            },
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                ],
            },
        ])
    )
    if caption:
        styled = styled.set_caption(caption)
    display(styled)


## 7. W&B login

In [ ]:
run = None
if USE_WANDB and WANDB_MODE != "disabled":
    import getpass
    import wandb

    os.environ["WANDB_MODE"] = WANDB_MODE
    os.environ.pop("WANDB_BASE_URL", None)
    if WANDB_MODE == "online":
        key = None
        try:
            from google.colab import userdata
            key = userdata.get("WANDB_API_KEY")
        except Exception:
            pass
        if key:
            wandb.login(key=key, relogin=True, verify=True)
        else:
            try:
                wandb.login(relogin=True, verify=True)
            except Exception:
                key = getpass.getpass("Paste W&B API key: ")
                wandb.login(key=key, relogin=True, verify=True)
        print("W&B login verified.")
    else:
        print("W&B mode:", WANDB_MODE)
else:
    print("W&B disabled.")

## 8. Training configuration — EDIT HERE

Defaults implement the planned repaired run. `RESUME_FROM` can point to the `*_latest.pt` file after a Colab interruption; optimizer state and epoch history will resume exactly.

In [ ]:
RUN_NAME = f"condor_set_transformer_{DATASET_TAG}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Data loading and split policy.
BATCH_SIZE = 2048  # reduce to 1024 only if GPU memory is insufficient
NUM_WORKERS = 0
SPLIT_SEED = 0  # sampler reproducibility; Sweep V3 partition is already frozen
LEGACY_HELDOUT_FRACTION = 0.30  # used only for old datasets without Data_Split
REPEAT_GROUP_NORMALIZATION = True

# Balanced training sampler. Validation and test remain natural/unweighted.
SAMPLER_MODE = "balanced"
SAMPLER_NATURAL_FRACTION = 0.50
SAMPLER_CONTEXT_FRACTION = 0.25
SAMPLER_PRIORITY_FRACTION = 0.25
SAMPLER_FEASIBLE_BOOST = 4.0
SAMPLER_TRACKING_BOUNDARY_BOOST = 2.0
SAMPLER_QOS_BOUNDARY_BOOST = 3.0

# Training operations.
MAX_EPOCHS = 250
BASE_LR = 3e-4
MIN_LR = 1e-6
WARMUP_EPOCHS = 5
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
EARLY_STOPPING_PATIENCE = 30
EARLY_STOPPING_MIN_DELTA = 1e-5

# Direct-label losses.
MEAN_TRACKING_LOSS_WEIGHT = 1.0
P90_TRACKING_LOSS_WEIGHT = 1.0
QOS_LOSS_WEIGHT = 4.0
TRACKING_BOUNDARY_MULTIPLIER = 2.0
QOS_BOUNDARY_MULTIPLIER = 2.0

# Checkpoint selection uses validation only. The test set is evaluated once
# after SELECTED_CHECKPOINT_ROLE is finalized.
RESTORE_ROLE_AFTER_TRAINING = "best_loss"
SELECTED_CHECKPOINT_ROLE = "best_feasibility"
RUN_FINAL_TEST_EVALUATION = True

CHECKPOINT_DIR = MODELS_DIR / RUN_NAME
CHECKPOINT_PREFIX = RUN_NAME
RESUME_FROM = None
# Example after interruption:
# RESUME_FROM = CHECKPOINT_DIR / f"{CHECKPOINT_PREFIX}_latest.pt"

MODEL_CONFIG = FlexDCBehaviorModelConfig(
    dim_job_mix=13,
    dim_dc_features=12,
    st_dim_hidden=512,
    st_num_heads=4,
    global_projection_dim=128,
    linear_dim_hidden=512,
    qos_projection_dim=256,
    skip_connections=True,
    layer_norm=True,
    include_masked_mean_pool=True,
)
CONSTANTS = FlexDCBehaviorConstants()

print("Run:", RUN_NAME)
print("Device:", DEVICE)
print("Dataset:", RESULTS_CSV)
print("Split mode:", SPLIT_MODE)
print("Repeat-group normalization:", REPEAT_GROUP_NORMALIZATION)
print("Resume from:", RESUME_FROM)
print("Model config:", MODEL_CONFIG.to_dict())


## 9. Prepare the preassigned split, standardize on training only, and audit seed groups

Sweep V3 already assigns every `Base_Plan_Row_ID` to train, validation, or test before simulation. This section verifies that assignment, keeps all seed repeats together, computes normalization statistics from training only, and constructs a base-configuration-aware training sampler.


In [ ]:
behavior_data = prepare_behavior_data(
    results_csv=RESULTS_CSV,
    diagnostics_csv=DIAGNOSTICS_CSV,
    batch_size=BATCH_SIZE,
    heldout_fraction=LEGACY_HELDOUT_FRACTION,
    split_seed=SPLIT_SEED,
    split_mode=SPLIT_MODE,
    split_column=SPLIT_COLUMN,
    num_workers=NUM_WORKERS,
    deduplicate=DEDUPLICATE,
    constants=CONSTANTS,
    sampler_mode=SAMPLER_MODE,
    sampler_seed=SPLIT_SEED,
    natural_fraction=SAMPLER_NATURAL_FRACTION,
    context_fraction=SAMPLER_CONTEXT_FRACTION,
    priority_fraction=SAMPLER_PRIORITY_FRACTION,
    feasible_boost=SAMPLER_FEASIBLE_BOOST,
    tracking_boundary_boost=SAMPLER_TRACKING_BOUNDARY_BOOST,
    qos_boundary_boost=SAMPLER_QOS_BOUNDARY_BOOST,
    repeat_group_normalization=REPEAT_GROUP_NORMALIZATION,
)

if V3_AUDIT is not None:
    expected_rows = int(V3_AUDIT.get("training_ready_rows", -1))
    if expected_rows >= 0 and behavior_data.metadata.original_row_count != expected_rows:
        raise RuntimeError(
            f"Loaded rows ({behavior_data.metadata.original_row_count}) do not match "
            f"the final audit ({expected_rows})."
        )

summary = pd.DataFrame([
    {"Item": "Original rows", "Value": behavior_data.metadata.original_row_count},
    {"Item": "Rows after cleanup", "Value": behavior_data.metadata.deduplicated_row_count},
    {"Item": "Training rows", "Value": behavior_data.metadata.train_row_count},
    {"Item": "Validation rows", "Value": behavior_data.metadata.validation_row_count},
    {"Item": "Test rows", "Value": behavior_data.metadata.test_row_count},
    {"Item": "Training base groups", "Value": behavior_data.metadata.train_group_count},
    {"Item": "Validation base groups", "Value": behavior_data.metadata.validation_group_count},
    {"Item": "Test base groups", "Value": behavior_data.metadata.test_group_count},
    {"Item": "Group overlap", "Value": behavior_data.audit["group_overlap"]},
    {"Item": "Repeated base groups", "Value": behavior_data.audit["repeated_base_groups"]},
    {"Item": "Training feasible rows", "Value": behavior_data.audit["train_actual_feasible_rows"]},
    {"Item": "Validation feasible rows", "Value": behavior_data.audit["validation_actual_feasible_rows"]},
    {"Item": "Test feasible rows", "Value": behavior_data.audit["test_actual_feasible_rows"]},
])
show_table(summary, "Sweep V3 preassigned split and seed-group audit", precision=4)

sampler_table = pd.DataFrame([
    {"Metric": key, "Value": value}
    for key, value in behavior_data.audit["sampler"].items()
    if isinstance(value, (int, float, bool))
])
show_table(sampler_table, "Base-configuration-aware balanced sampler", precision=6)

context_split = pd.DataFrame(behavior_data.audit["context_split_summary"])
show_table(context_split, "Per-context preassigned split", precision=0)

print("Split strategy:", behavior_data.metadata.split_strategy)
print("Token features:", behavior_data.metadata.token_feature_names)
print("Global features:", behavior_data.metadata.global_feature_names)
print("Direct labels:", behavior_data.metadata.direct_label_names)
print(
    "Test set is loaded but will not be evaluated until the final selected "
    "checkpoint section."
)


## 10. Create the repaired CONDOR behavior model

In [ ]:
model = DataCenterBehaviorModel(MODEL_CONFIG)
print(model)
print(f"Parameters: {model.parameter_count():,}")

# One real batch shape check before W&B/training.
real_batch = next(iter(behavior_data.train_loader))
with torch.no_grad():
    smoke_output = model(real_batch["features"], real_batch["workload"], real_batch["mask"])
print("Global input shape:", tuple(real_batch["features"].shape))
print("Token input shape:", tuple(real_batch["workload"].shape))
print("Tracking output shape:", tuple(smoke_output["tracking_logs"].shape))
print("QoS output shape:", tuple(smoke_output["qos_probabilities"].shape))

## 11. Start W&B run

In [ ]:
training_config = {
    "model_version": "v3",
    "dataset_profile": DATASET_PROFILE,
    "dataset_tag": DATASET_TAG,
    "results_csv": str(RESULTS_CSV),
    "diagnostics_csv": str(DIAGNOSTICS_CSV) if DIAGNOSTICS_CSV else None,
    "final_audit_json": str(FINAL_AUDIT_JSON) if FINAL_AUDIT_JSON else None,
    "batch_size": BATCH_SIZE,
    "split_mode": SPLIT_MODE,
    "split_column": SPLIT_COLUMN,
    "split_seed": SPLIT_SEED,
    "train_rows": behavior_data.metadata.train_row_count,
    "validation_rows": behavior_data.metadata.validation_row_count,
    "test_rows": behavior_data.metadata.test_row_count,
    "repeat_group_normalization": REPEAT_GROUP_NORMALIZATION,
    "sampler_mode": SAMPLER_MODE,
    "sampler_natural_fraction": SAMPLER_NATURAL_FRACTION,
    "sampler_context_fraction": SAMPLER_CONTEXT_FRACTION,
    "sampler_priority_fraction": SAMPLER_PRIORITY_FRACTION,
    "sampler_feasible_boost": SAMPLER_FEASIBLE_BOOST,
    "sampler_tracking_boundary_boost": SAMPLER_TRACKING_BOUNDARY_BOOST,
    "sampler_qos_boundary_boost": SAMPLER_QOS_BOUNDARY_BOOST,
    "max_epochs": MAX_EPOCHS,
    "base_lr": BASE_LR,
    "min_lr": MIN_LR,
    "warmup_epochs": WARMUP_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "gradient_clip_norm": GRADIENT_CLIP_NORM,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "mean_tracking_loss_weight": MEAN_TRACKING_LOSS_WEIGHT,
    "p90_tracking_loss_weight": P90_TRACKING_LOSS_WEIGHT,
    "qos_loss_weight": QOS_LOSS_WEIGHT,
    "tracking_boundary_multiplier": TRACKING_BOUNDARY_MULTIPLIER,
    "qos_boundary_multiplier": QOS_BOUNDARY_MULTIPLIER,
    "device": DEVICE,
    "model_config": MODEL_CONFIG.to_dict(),
    "direct_labels": behavior_data.metadata.direct_label_names,
    "token_features": behavior_data.metadata.token_feature_names,
    "global_features": behavior_data.metadata.global_feature_names,
}

if USE_WANDB and WANDB_MODE != "disabled":
    import wandb

    resume_wandb_id = None
    if RESUME_FROM and Path(RESUME_FROM).exists():
        resume_payload = torch.load(RESUME_FROM, map_location="cpu", weights_only=False)
        resume_wandb_id = resume_payload.get("wandb_run_id")

    run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=RUN_NAME,
        mode=WANDB_MODE,
        config=training_config,
        id=resume_wandb_id,
        resume="must" if resume_wandb_id else None,
    )
    run.summary["data/train_rows"] = behavior_data.metadata.train_row_count
    run.summary["data/validation_rows"] = behavior_data.metadata.validation_row_count
    run.summary["data/test_rows"] = behavior_data.metadata.test_row_count
    run.summary["data/train_base_groups"] = behavior_data.metadata.train_group_count
    run.summary["data/validation_base_groups"] = behavior_data.metadata.validation_group_count
    run.summary["data/test_base_groups"] = behavior_data.metadata.test_group_count
    run.summary["data/group_overlap"] = behavior_data.audit["group_overlap"]
    run.summary["data/repeated_base_groups"] = behavior_data.audit["repeated_base_groups"]
    print("W&B run:", run.url if hasattr(run, "url") else run.id)
else:
    run = None
    print("W&B run disabled.")


## 12. Train with scheduling, clipping, early stopping, and resumable checkpoints

A checkpoint is saved at every epoch before the next epoch starts. A browser disconnect does not lose the current run, and a terminated runtime can resume from `*_latest.pt`.

In [ ]:
model, training_result = train_behavior_model(
    model,
    behavior_data,
    epochs=MAX_EPOCHS,
    base_lr=BASE_LR,
    min_lr=MIN_LR,
    warmup_epochs=WARMUP_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    gradient_clip_norm=GRADIENT_CLIP_NORM,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    device_name=DEVICE,
    mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
    p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,
    qos_weight=QOS_LOSS_WEIGHT,
    tracking_boundary_multiplier=TRACKING_BOUNDARY_MULTIPLIER,
    qos_boundary_multiplier=QOS_BOUNDARY_MULTIPLIER,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_prefix=CHECKPOINT_PREFIX,
    model_config=MODEL_CONFIG.to_dict(),
    training_config=training_config,
    resume_from=RESUME_FROM,
    restore_role=RESTORE_ROLE_AFTER_TRAINING,
    wandb_run=run,
    metrics_every_n_epochs=1,
    verbose=True,
)

history = training_result.history
HISTORY_CSV = RESULTS_DIR / f"{RUN_NAME}_history.csv"
history.to_csv(HISTORY_CSV, index=False)

checkpoint_table = pd.DataFrame([
    {"Role": role, "Path": path, "Exists": Path(path).exists()}
    for role, path in training_result.checkpoint_paths.items()
])
show_table(checkpoint_table, "Saved checkpoints")
print("Training summary:")
print(json.dumps(training_result.summary, indent=2))
print("History:", HISTORY_CSV)

## 13. Compare every saved checkpoint on the validation split only

Validation chooses the checkpoint. The test loader is not called in this section.


In [ ]:
device = choose_device(DEVICE)
comparison_rows = []
checkpoint_metrics = {}

# Checkpoint selection is validation-only. The test loader is never called here.
for role in ["best_loss", "best_objective", "best_feasibility", "final"]:
    path = Path(training_result.checkpoint_paths[role])
    if not path.exists():
        continue
    candidate, checkpoint = load_behavior_model_checkpoint(
        path,
        model_class=DataCenterBehaviorModel,
        config_class=FlexDCBehaviorModelConfig,
        device_name=DEVICE,
    )
    validation_metrics, _ = evaluate_behavior_loader(
        candidate,
        behavior_data.validation_loader,
        device=device,
        metadata=behavior_data.metadata,
        prefix="validation",
        mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
        p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,
        qos_weight=QOS_LOSS_WEIGHT,
        tracking_boundary_multiplier=TRACKING_BOUNDARY_MULTIPLIER,
        qos_boundary_multiplier=QOS_BOUNDARY_MULTIPLIER,
        return_rows=False,
        include_workload_metrics=True,
    )
    checkpoint_metrics[role] = validation_metrics
    comparison_rows.append({
        "Role": role,
        "Epoch": checkpoint.get("epoch"),
        "Validation loss": validation_metrics["validation/loss/total"],
        "Objective R2": validation_metrics["validation/cost/full_objective/r2"],
        "Objective Spearman": validation_metrics["validation/cost/full_objective/spearman"],
        "P90 physical R2": validation_metrics["validation/tracking/p90_physical/r2"],
        "Per-job Pj MAE": validation_metrics["validation/qos/per_job_probability/mae"],
        "Max-Pj MAE": validation_metrics["validation/qos/max_probability/mae"],
        "Feasible precision": validation_metrics["validation/feasibility/combined/feasible_precision"],
        "Feasible recall": validation_metrics["validation/feasibility/combined/actual_feasible_accuracy"],
        "Feasible F1": validation_metrics["validation/feasibility/combined/f1_feasible"],
        "False-feasible rate": validation_metrics["validation/feasibility/combined/false_feasible_rate"],
    })

checkpoint_comparison = pd.DataFrame(comparison_rows)
show_table(checkpoint_comparison, "Validation-only comparison of saved checkpoints", precision=5)
CHECKPOINT_COMPARISON_CSV = RESULTS_DIR / f"{RUN_NAME}_checkpoint_comparison.csv"
checkpoint_comparison.to_csv(CHECKPOINT_COMPARISON_CSV, index=False)

if run is not None:
    import wandb
    run.log({"validation_checkpoint_comparison": wandb.Table(dataframe=checkpoint_comparison)})

print("Configured selected role:", SELECTED_CHECKPOINT_ROLE)
print("The untouched test split has not been evaluated.")


## 14. Lock the selected checkpoint and evaluate train + validation

After this cell, `SELECTED_CHECKPOINT_ROLE` is treated as final for the one-time test evaluation.


In [ ]:
selected_path = Path(training_result.checkpoint_paths.get(SELECTED_CHECKPOINT_ROLE, ""))
if not selected_path.exists():
    print(f"{SELECTED_CHECKPOINT_ROLE} is unavailable; falling back to best_loss.")
    SELECTED_CHECKPOINT_ROLE = "best_loss"
    selected_path = Path(training_result.checkpoint_paths[SELECTED_CHECKPOINT_ROLE])

model, selected_checkpoint = load_behavior_model_checkpoint(
    selected_path,
    model_class=DataCenterBehaviorModel,
    config_class=FlexDCBehaviorModelConfig,
    device_name=DEVICE,
)

metrics, train_predictions, validation_predictions = final_behavior_evaluation(
    model,
    behavior_data,
    device_name=DEVICE,
    mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
    p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,
    qos_weight=QOS_LOSS_WEIGHT,
    tracking_boundary_multiplier=TRACKING_BOUNDARY_MULTIPLIER,
    qos_boundary_multiplier=QOS_BOUNDARY_MULTIPLIER,
)
validation_context_summary = context_metrics_table(validation_predictions)

METRICS_CSV = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_train_validation_metrics.csv"
TRAIN_PREDICTIONS_CSV = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_train_predictions.csv"
VALIDATION_PREDICTIONS_CSV = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_validation_predictions.csv"
VALIDATION_CONTEXT_SUMMARY_CSV = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_validation_context_summary.csv"

pd.DataFrame([metrics]).to_csv(METRICS_CSV, index=False)
train_predictions.to_csv(TRAIN_PREDICTIONS_CSV, index=False)
validation_predictions.to_csv(VALIDATION_PREDICTIONS_CSV, index=False)
validation_context_summary.to_csv(VALIDATION_CONTEXT_SUMMARY_CSV, index=False)

key_metrics = pd.DataFrame([
    {"Metric": "Selected role", "Value": SELECTED_CHECKPOINT_ROLE},
    {"Metric": "Selected epoch", "Value": selected_checkpoint.get("epoch")},
    {"Metric": "Validation objective R2", "Value": metrics["validation/cost/full_objective/r2"]},
    {"Metric": "Validation objective Spearman", "Value": metrics["validation/cost/full_objective/spearman"]},
    {"Metric": "Validation p90 physical R2", "Value": metrics["validation/tracking/p90_physical/r2"]},
    {"Metric": "Validation per-job Pj MAE", "Value": metrics["validation/qos/per_job_probability/mae"]},
    {"Metric": "Validation max-Pj MAE", "Value": metrics["validation/qos/max_probability/mae"]},
    {"Metric": "Validation feasible precision", "Value": metrics["validation/feasibility/combined/feasible_precision"]},
    {"Metric": "Validation feasible recall", "Value": metrics["validation/feasibility/combined/actual_feasible_accuracy"]},
    {"Metric": "Validation feasible F1", "Value": metrics["validation/feasibility/combined/f1_feasible"]},
    {"Metric": "Validation false-feasible rate", "Value": metrics["validation/feasibility/combined/false_feasible_rate"]},
])
show_table(key_metrics, "Locked checkpoint — train/validation evaluation", precision=5)
show_table(
    validation_context_summary,
    "Validation feasibility and error by context",
    precision=5,
)

print("Selected checkpoint is now locked:", selected_path)
print("The next section evaluates the untouched test split once.")


## 15. One-time untouched test evaluation

This cell evaluates **only the already locked checkpoint**. The test set was not used for:

- gradient updates;
- normalization statistics;
- learning-rate or loss decisions;
- early stopping;
- checkpoint comparison or selection.

Do not use these test results to change the current run and then report the same test set as untouched. If you redesign the model after reading these results, the test becomes development feedback and a new final test is required.


In [ ]:
TEST_METRICS_CSV = None
TEST_PREDICTIONS_CSV = None
TEST_CONTEXT_SUMMARY_CSV = None
test_metrics = {}
test_predictions = pd.DataFrame()
test_context_summary = pd.DataFrame()

if RUN_FINAL_TEST_EVALUATION:
    test_metrics, test_predictions = final_behavior_test_evaluation(
        model,
        behavior_data,
        device_name=DEVICE,
        mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
        p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,
        qos_weight=QOS_LOSS_WEIGHT,
        tracking_boundary_multiplier=TRACKING_BOUNDARY_MULTIPLIER,
        qos_boundary_multiplier=QOS_BOUNDARY_MULTIPLIER,
    )
    test_context_summary = context_metrics_table(test_predictions)

    TEST_METRICS_CSV = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_test_metrics.csv"
    TEST_PREDICTIONS_CSV = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_test_predictions.csv"
    TEST_CONTEXT_SUMMARY_CSV = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_test_context_summary.csv"

    pd.DataFrame([test_metrics]).to_csv(TEST_METRICS_CSV, index=False)
    test_predictions.to_csv(TEST_PREDICTIONS_CSV, index=False)
    test_context_summary.to_csv(TEST_CONTEXT_SUMMARY_CSV, index=False)

    test_key_metrics = pd.DataFrame([
        {"Metric": "Test objective R2", "Value": test_metrics["test/cost/full_objective/r2"]},
        {"Metric": "Test objective Spearman", "Value": test_metrics["test/cost/full_objective/spearman"]},
        {"Metric": "Test p90 physical R2", "Value": test_metrics["test/tracking/p90_physical/r2"]},
        {"Metric": "Test per-job Pj MAE", "Value": test_metrics["test/qos/per_job_probability/mae"]},
        {"Metric": "Test max-Pj MAE", "Value": test_metrics["test/qos/max_probability/mae"]},
        {"Metric": "Test feasible precision", "Value": test_metrics["test/feasibility/combined/feasible_precision"]},
        {"Metric": "Test feasible recall", "Value": test_metrics["test/feasibility/combined/actual_feasible_accuracy"]},
        {"Metric": "Test feasible F1", "Value": test_metrics["test/feasibility/combined/f1_feasible"]},
        {"Metric": "Test false-feasible rate", "Value": test_metrics["test/feasibility/combined/false_feasible_rate"]},
    ])
    show_table(test_key_metrics, "Untouched test-set results", precision=5)
    show_table(test_context_summary, "Untouched test results by context", precision=5)
    print("One-time test evaluation complete.")
else:
    print("Final test evaluation is disabled. Set RUN_FINAL_TEST_EVALUATION=True after checkpoint selection is locked.")


## 16. Send final metrics and tables to W&B

In [ ]:
if run is not None:
    import wandb

    for key, value in metrics.items():
        if isinstance(value, (int, float, np.integer, np.floating)) and np.isfinite(value):
            run.summary[f"selected/{key}"] = float(value)
    for key, value in test_metrics.items():
        if isinstance(value, (int, float, np.integer, np.floating)) and np.isfinite(value):
            run.summary[f"final/{key}"] = float(value)

    run.summary["selected_checkpoint_role"] = SELECTED_CHECKPOINT_ROLE
    run.summary["selected_checkpoint_epoch"] = int(selected_checkpoint.get("epoch", -1))
    run.summary["selected_checkpoint_path"] = str(selected_path)
    run.summary["test_evaluated_after_checkpoint_lock"] = bool(RUN_FINAL_TEST_EVALUATION)

    log_payload = {
        "validation_context_summary": wandb.Table(dataframe=validation_context_summary),
        "validation_prediction_sample": wandb.Table(
            dataframe=sample_prediction_rows(validation_predictions, max_rows=2048, seed=0)
        ),
    }
    if RUN_FINAL_TEST_EVALUATION:
        log_payload.update({
            "final_test_context_summary": wandb.Table(dataframe=test_context_summary),
            "final_test_prediction_sample": wandb.Table(
                dataframe=sample_prediction_rows(test_predictions, max_rows=2048, seed=0)
            ),
        })
    run.log(log_payload)
    print("W&B validation and final-test summaries updated.")
else:
    print("W&B disabled.")


## 17. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

curves = [
    ("validation/loss/total", "Validation weighted loss"),
    ("learning_rate", "Learning rate"),
    ("validation/cost/full_objective/r2", "Validation objective R²"),
    ("validation/feasibility/combined/feasible_precision", "Validation feasible precision"),
    ("validation/feasibility/combined/actual_feasible_accuracy", "Validation feasible recall"),
    ("validation/by_family/W2/feasibility/feasible_precision", "W2 validation feasible precision"),
    ("validation/by_family/W2/feasibility/actual_feasible_accuracy", "W2 validation feasible recall"),
]
for column, title in curves:
    if column not in history.columns:
        continue
    plt.figure(figsize=(8, 4))
    plt.plot(history["epoch"], history[column])
    plt.xlabel("Epoch")
    plt.ylabel(title)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()


## 18. Package artifacts and optionally download

In [ ]:
import zipfile

ARTIFACT_ZIP = RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_artifacts.zip"
files_to_package = [
    HISTORY_CSV,
    CHECKPOINT_COMPARISON_CSV,
    METRICS_CSV,
    TRAIN_PREDICTIONS_CSV,
    VALIDATION_PREDICTIONS_CSV,
    VALIDATION_CONTEXT_SUMMARY_CSV,
    selected_path,
    MODEL_PY,
    UTILS_PY,
    TEST_PY,
]
if FINAL_AUDIT_JSON is not None:
    files_to_package.append(FINAL_AUDIT_JSON)
for optional_path in [
    TEST_METRICS_CSV,
    TEST_PREDICTIONS_CSV,
    TEST_CONTEXT_SUMMARY_CSV,
]:
    if optional_path is not None:
        files_to_package.append(optional_path)

# Also include all checkpoint roles so later analysis can compare or resume.
for path in training_result.checkpoint_paths.values():
    p = Path(path)
    if p.exists() and p not in files_to_package:
        files_to_package.append(p)

with zipfile.ZipFile(ARTIFACT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_package:
        path = Path(path)
        if path.exists():
            archive.write(path, arcname=path.name)

print("Artifact ZIP:", ARTIFACT_ZIP)
print("Size MB:", ARTIFACT_ZIP.stat().st_size / 1024**2)

if run is not None:
    run.finish()

if RUN_ENV == "colab":
    try:
        from google.colab import files
        # Uncomment to download automatically:
        # files.download(str(ARTIFACT_ZIP))
        print("Use files.download(str(ARTIFACT_ZIP)) to download.")
    except Exception:
        pass
